# Notebook B · Custom PyramidKV-style eviction and paired comparison

Run the paper’s custom attention wrapper and eviction code on the exact 200 TREC inputs exported by Notebook A. The comparison includes stock FullKV in both environments and reports paired accuracy differences, 95% bootstrap intervals, and the realized cache sizes.

Open this notebook in a **fresh** Colab runtime with **T4 GPU** and **Runtime version 2026.07**. Run step 1, upload the Notebook A ZIP if prompted, then choose **Runtime → Restart session**. After Colab reconnects, run steps 2 through 5. The uploaded ZIP stays available in `/content`. The checkpoint, precision, nominal budget, input tokens and stop tokens come from that ZIP. [Colab runtime-version guide](https://research.google.com/colaboratory/runtime-version-faq.html)

## 1. Register Notebook A’s ZIP and install the custom environment

The setup pins the NumPy, SciPy and scikit-learn versions together and tests them in a clean Python process. When the cell says setup passed, choose **Runtime → Restart session**. After reconnecting, continue from step 2 in the same Colab session.

In [1]:
import hashlib
import json
import os
import subprocess
import sys
from importlib import metadata
from pathlib import Path
from zipfile import ZipFile

from google.colab import files

# Leave blank to use the saved upload or choose a ZIP from your computer.
OFFICIAL_ARCHIVE = ""
SETUP_DIR = Path("/content/paper744_trec_notebook_b")
SETUP_FILE = SETUP_DIR / "setup_v2.json"
SETUP_DIR.mkdir(exist_ok=True)
saved_setup = json.loads(SETUP_FILE.read_text()) if SETUP_FILE.is_file() else {}
assert sys.version_info[:2] == (3, 12), (
    "In Colab choose Runtime > Change runtime type > Runtime version 2026.07 (Python 3.12)."
)
gpu_names = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True,
).splitlines()
assert len(gpu_names) == 1 and "T4" in gpu_names[0], f"Expected one NVIDIA T4, found {gpu_names}."

if OFFICIAL_ARCHIVE:
    reference_zip = Path(OFFICIAL_ARCHIVE)
elif saved_setup and Path(saved_setup["archive"]).is_file():
    reference_zip = Path(saved_setup["archive"])
else:
    existing = sorted(Path("/content").glob("paper744_trec_official_*.zip"))
    if len(existing) == 1:
        reference_zip = existing[0]
    else:
        uploaded = files.upload()
        assert len(uploaded) == 1, "Select exactly one Notebook A ZIP."
        reference_zip = Path(next(iter(uploaded)))
        del uploaded
reference_zip = reference_zip.resolve()
assert reference_zip.is_file() and reference_zip.suffix == ".zip"
with ZipFile(reference_zip) as archive:
    assert archive.getinfo("workload.json").file_size < 100_000
    settings = json.loads(archive.read("workload.json"))
assert settings["schema"] == "paper744-llama-3b-trec-v1", "Upload the ZIP from Notebook A."
precision = settings["precision"]
assert precision in ("awq", "fp16")
archive_sha = hashlib.sha256(reference_zip.read_bytes()).hexdigest()
print(f"Using {reference_zip.name} ({precision}).")

# GPTQModel 7.1.0 requires NumPy 2.2.6 on Python 3.12.
requirements = {
    "numpy": "2.2.6",
    "scipy": "1.16.3",
    "scikit-learn": "1.6.1",
    "transformers": "5.12.1",
    "accelerate": "1.13.0",
}
if precision == "awq":
    requirements.update({"autoawq": "0.2.9", "gptqmodel": "7.1.0", "torchao": "0.17.0"})


def installed_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None


# Keep Colab's matched CUDA wheels while installing notebook dependencies.
torch_stack = {
    name: version
    for name in ("torch", "torchvision", "torchaudio", "triton")
    if (version := installed_version(name)) is not None
}
assert "torch" in torch_stack and "triton" in torch_stack, "Select a Colab T4 runtime."
expected = {**requirements, **torch_stack}
ready = (
    saved_setup.get("archive") == str(reference_zip)
    and saved_setup.get("archive_sha256") == archive_sha
    and saved_setup.get("versions") == expected
    and all(installed_version(name) == version for name, version in expected.items())
)
if not ready:
    constraints = SETUP_DIR / "torch_constraints.txt"
    constraints.write_text("".join(f"{name}=={version}\n" for name, version in torch_stack.items()))
    pip = [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check"]
    subprocess.run([
        *pip, "--constraint", str(constraints),
        *(f"{name}=={version}" for name, version in requirements.items()),
    ], check=True)
    assert all(installed_version(name) == version for name, version in expected.items())
    # Run imports in a clean interpreter before the notebook kernel restarts.
    # This catches on-disk package problems separately from stale in-memory NumPy modules.
    probe = """
import sys
import numpy as np
import scipy
import sklearn
from scipy.sparse import csr_matrix
from sklearn.metrics import roc_curve
import torch
import transformers
from transformers import AutoModelForCausalLM, DynamicCache, PreTrainedTokenizerFast
from transformers.modeling_utils import ALL_ATTENTION_FUNCTIONS
from transformers.models.llama.modeling_llama import (
    LlamaAttention, apply_rotary_pos_emb, eager_attention_forward, repeat_kv,
)
assert np.strings.center(np.array(["x"]), 3)[0] == " x "
assert csr_matrix(np.eye(2)).nnz == 2
assert roc_curve([0, 1], [0.0, 1.0])[1][-1] == 1
assert torch.cuda.is_available() and "T4" in torch.cuda.get_device_name()
if sys.argv[1] == "awq":
    from gptqmodel import BACKEND
print(f"Clean-process imports passed: NumPy {np.__version__}, SciPy {scipy.__version__}, "
      f"scikit-learn {sklearn.__version__}, Transformers {transformers.__version__}, "
      f"PyTorch {torch.__version__}.")
"""
    subprocess.run([sys.executable, "-c", probe, precision], check=True)
    saved_setup = {
        "archive": str(reference_zip),
        "archive_sha256": archive_sha,
        "precision": precision,
        "versions": expected,
        "install_pid": os.getpid(),
    }
    temporary = SETUP_FILE.with_suffix(".tmp")
    temporary.write_text(json.dumps(saved_setup))
    temporary.replace(SETUP_FILE)

if saved_setup["install_pid"] == os.getpid():
    print("Setup passed. Choose Runtime > Restart session, then start from step 2.")
else:
    print("Setup and restart are complete. Continue with step 2.")

Using paper744_trec_official_20261122T183143Z-faaf2ef3.zip (awq).
Setup and restart are complete. Continue with step 2.


## 2. Resume after the restart and verify the input archive

Start here after restarting the session. The notebook recovers the saved ZIP path and precision, then validates the checksums, model revision, case alignment, and raw predictions.

In [2]:
import hashlib
import json
import os
from importlib import metadata
from pathlib import Path

SETUP_FILE = Path("/content/paper744_trec_notebook_b/setup_v2.json")
if not SETUP_FILE.is_file():
    raise RuntimeError("Run step 1 to install the packages and register Notebook A's ZIP.")
setup = json.loads(SETUP_FILE.read_text())
if setup["install_pid"] == os.getpid():
    raise RuntimeError("Choose Runtime > Restart session after step 1, then resume here at step 2.")
REFERENCE_ZIP = Path(setup["archive"])
PRECISION = setup["precision"]
if not REFERENCE_ZIP.is_file() or hashlib.sha256(REFERENCE_ZIP.read_bytes()).hexdigest() != setup["archive_sha256"]:
    raise RuntimeError("Notebook A's ZIP changed or is missing. Run step 1 to register it again.")
for package, expected in setup["versions"].items():
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = None
    if installed != expected:
        raise RuntimeError(f"Expected {package} {expected}, found {installed}. Rerun step 1.")

In [3]:
import ast
import importlib.metadata
import platform
import subprocess
import time
import uuid
from datetime import datetime, timezone
from shutil import copyfile
from tempfile import TemporaryDirectory
from urllib.request import urlopen
from zipfile import ZIP_DEFLATED, ZipFile

import numpy as np
import torch
import transformers
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, DynamicCache, PreTrainedTokenizerFast

SCHEMA = "paper744-llama-3b-trec-v1"
UPSTREAM_COMMIT = "68cd9551a63fedf362ddb0edb008badd04c24996"
UPSTREAM_URL = "https://github.com/Zefan-Cai/KVCache-Factory.git"
UPSTREAM_RAW = f"https://raw.githubusercontent.com/Zefan-Cai/KVCache-Factory/{UPSTREAM_COMMIT}/"
DATA_PATH = "data/LongBench/trec.jsonl"
DATA_SHA256 = "20a88e14f7442f58a39082cdd7396f1093ea8162da0a4ea7ead7d6757514c8f4"
METRICS_SHA256 = "76cf741219c29ae5e8fd0c1b1c00397f53aa14a32177f0867f1a4ed03aa77f24"
MODEL_SPECS = {
    "awq": ("casperhansen/llama-3.2-3b-instruct-awq", "272b3bde867b606760447deb9a4d2719fbdfd3ae"),
    "fp16": ("unsloth/Llama-3.2-3B-Instruct", "006f5dcd1393c3add266de40994ba96225e9689d"),
}
NUM_LAYERS = 28
NUM_Q_HEADS = 24
NUM_KV_HEADS = 8
HEAD_DIM = 128
CASE_COUNT = 200
MAX_NEW_TOKENS = 64
SEED = 407
ATTENTION_BACKEND = "sdpa"


def digest(data):
    return hashlib.sha256(data).hexdigest()


def object_digest(value):
    encoded = json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()
    return digest(encoded)


def id_digest(ids):
    return digest(",".join(map(str, ids)).encode("ascii"))


def download_upstream(name, expected_sha256):
    with urlopen(UPSTREAM_RAW + name, timeout=60) as response:
        data = response.read()
    assert digest(data) == expected_sha256, f"Unexpected source hash for {name}"
    return data


def write_json(path, value):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8")
    temporary.replace(path)


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def read_jsonl(data):
    return [json.loads(line) for line in data.splitlines() if line.strip()]


def write_jsonl(path, rows):
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as out:
        for row in rows:
            out.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + "\n")
    temporary.replace(path)


def start_output_directory(arm):
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run_id = f"{stamp}-{uuid.uuid4().hex[:8]}"
    output = Path("/content") / f"paper744_trec_{arm}_{run_id}"
    output.mkdir()
    return output, run_id


def package_versions():
    values = {"python": platform.python_version(), "torch": str(torch.__version__), "cuda": torch.version.cuda}
    for name in ("transformers", "tokenizers", "accelerate", "autoawq", "gptqmodel", "triton",
                 "numpy", "scipy", "scikit-learn", "torchao"):
        try:
            values[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            values[name] = None
    return values


def observed_gpu():
    assert torch.cuda.is_available() and torch.cuda.device_count() == 1
    device = torch.cuda.get_device_properties(0)
    assert "T4" in device.name, f"Select a Colab T4 runtime, found {device.name}"
    return {"name": device.name, "memory_mib": device.total_memory // 2**20}


def tokenizer_and_stop_ids(spec):
    names = ("config.json", "tokenizer.json", "tokenizer_config.json", "special_tokens_map.json")
    snapshot = Path(snapshot_download(
        spec["model_id"], revision=spec["model_revision"], allow_patterns=list(names),
    ))
    if transformers.__version__ == "4.44.2":
        # Tokenizers 0.19 reads BPE merges as strings. Convert their serialization
        # without changing the ordered merge pairs, vocabulary, or tokenizer settings.
        with TemporaryDirectory() as directory:
            local = Path(directory)
            for name in ("tokenizer_config.json", "special_tokens_map.json"):
                if (snapshot / name).is_file():
                    copyfile(snapshot / name, local / name)
            tokenizer_data = read_json(snapshot / "tokenizer.json")
            bpe = tokenizer_data["model"]
            assert bpe["type"] == "BPE"
            bpe["merges"] = [" ".join(pair) if isinstance(pair, list) else pair for pair in bpe["merges"]]
            write_json(local / "tokenizer.json", tokenizer_data)
            tokenizer = PreTrainedTokenizerFast.from_pretrained(str(local))
    else:
        tokenizer = PreTrainedTokenizerFast.from_pretrained(str(snapshot))
    model_eos = read_json(snapshot / "config.json")["eos_token_id"]
    if isinstance(model_eos, int):
        model_eos = [model_eos]
    eot = tokenizer.convert_tokens_to_ids("<|eot_id|>")
    assert eot is not None and eot != tokenizer.unk_token_id
    stop_tokens = list(dict.fromkeys([*model_eos, tokenizer.eos_token_id, eot]))
    assert all(type(token) is int and 0 <= token < len(tokenizer) for token in stop_tokens)
    return tokenizer, stop_tokens


def validate_workload(spec, cases):
    assert spec["schema"] == SCHEMA and spec["dataset"] == "LongBench v1 TREC"
    assert spec["dataset_sha256"] == DATA_SHA256
    assert spec["upstream_commit"] == UPSTREAM_COMMIT and spec["metrics_sha256"] == METRICS_SHA256
    assert spec["attention_backend"] == ATTENTION_BACKEND and spec["max_new_tokens"] == MAX_NEW_TOKENS
    assert tuple((spec["model_id"], spec["model_revision"])) == MODEL_SPECS[spec["precision"]]
    assert len(cases) == spec["case_count"] == CASE_COUNT
    assert spec["batch_size"] == 1 and spec["decoding"] == "greedy"
    assert spec["stop_token_ids"] and len(set(spec["stop_token_ids"])) == len(spec["stop_token_ids"])
    assert all(type(token) is int and 0 <= token < spec["vocab_size"]
               for token in [*spec["stop_token_ids"], spec["pad_token_id"]])
    assert [case["index"] for case in cases] == list(range(CASE_COUNT))
    assert len({case["case_id"] for case in cases}) == CASE_COUNT
    assert 32 < spec["budget"] < spec["prompt_token_cap"] <= 7500
    assert all(case["answers"] and set(case["answers"]) <= set(case["all_classes"]) for case in cases)
    for case in cases:
        ids = case["input_ids"]
        assert 0 < len(ids) <= spec["prompt_token_cap"]
        assert all(type(token) is int and 0 <= token < spec["vocab_size"] for token in ids)
        assert id_digest(ids) == case["input_ids_sha256"]


def load_model(spec, expected_transformers):
    assert transformers.__version__ == expected_transformers, "Restart the runtime and rerun setup"
    torch.manual_seed(SEED)
    model = AutoModelForCausalLM.from_pretrained(
        spec["model_id"], revision=spec["model_revision"],
        torch_dtype=torch.float16, device_map={"": 0}, low_cpu_mem_usage=True,
        attn_implementation=ATTENTION_BACKEND, use_safetensors=True,
    ).eval()
    config = model.config
    assert (config.model_type, config.num_hidden_layers, config.num_attention_heads,
            config.num_key_value_heads) == ("llama", NUM_LAYERS, NUM_Q_HEADS, NUM_KV_HEADS)
    assert getattr(config, "head_dim", config.hidden_size // NUM_Q_HEADS) == HEAD_DIM
    assert config.vocab_size == spec["vocab_size"]
    assert config._attn_implementation == ATTENTION_BACKEND
    if spec["precision"] == "awq":
        assert config.quantization_config.to_dict()["quant_method"] == "awq"
        assert config.quantization_config.to_dict()["bits"] == 4
        assert hasattr(model.model.layers[0].self_attn.q_proj, "qweight")
    else:
        assert model.model.layers[0].self_attn.q_proj.weight.dtype == torch.float16
    assert model.get_input_embeddings().weight.device.type == "cuda", "Expected the 3B weights on the T4"
    return model


def model_runtime(model):
    projection = model.model.layers[0].self_attn.q_proj
    cls = type(projection)
    return {"q_proj_class": f"{cls.__module__}.{cls.__name__}", "model_dtype": str(model.dtype)}


def cache_shapes(cache):
    if hasattr(cache, "layers"):  # Transformers 5.x DynamicCache
        layers = [layer.keys for layer in cache.layers]
    elif hasattr(cache, "key_cache"):  # Transformers 4.44 DynamicCache
        layers = cache.key_cache
    else:
        layers = [pair[0] for pair in cache]
    return [(int(key.shape[1]), int(key.shape[-2])) for key in layers]


def build_row(case, mode, generated_ids, kept, final, tokenizer, spec):
    assert generated_ids and len(generated_ids) <= spec["max_new_tokens"]
    assert all(0 <= token < spec["vocab_size"] for token in generated_ids)
    assert len(kept) == len(final) == NUM_LAYERS
    prompt_len = len(case["input_ids"])
    assert all(0 < length <= prompt_len for _, length in kept)
    assert all(heads in (NUM_KV_HEADS, NUM_Q_HEADS) for heads, _ in kept)
    assert all(
        head == final_head and final_len == kept_len + len(generated_ids) - 1
        for (head, kept_len), (final_head, final_len) in zip(kept, final)
    ), "The observed prefill cache was not used through decode"
    if mode == "fullkv":
        assert all(shape == (NUM_KV_HEADS, prompt_len) for shape in kept)
    elif prompt_len > spec["budget"]:
        assert any(length < prompt_len for _, length in kept), "Eviction did not reduce the stored cache"
    return {
        "case_id": case["case_id"], "index": case["index"], "mode": mode,
        "input_ids_sha256": case["input_ids_sha256"],
        "generated_ids": generated_ids,
        "prediction": tokenizer.decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False),
        "prefill_cache_tokens": [length for _, length in kept],
        "cache_heads": [heads for heads, _ in kept],
        "final_cache_tokens": [length for _, length in final],
    }


def pilot_cases(cases):
    by_length = sorted(cases, key=lambda case: (len(case["input_ids"]), case["original_token_count"], case["index"]))
    return [by_length[i] for i in (0, len(cases)//4, len(cases)//2, (3*len(cases))//4, len(cases)-1)]


def validate_rows(rows, cases, mode, spec):
    assert len(rows) == len(cases)
    assert [r["case_id"] for r in rows] == [c["case_id"] for c in cases]
    for row, case in zip(rows, cases):
        assert row["mode"] == mode and row["index"] == case["index"]
        assert row["input_ids_sha256"] == case["input_ids_sha256"]
        prompt_len = len(case["input_ids"])
        generated = row["generated_ids"]
        assert len(generated) <= spec["max_new_tokens"] and len(generated) > 0
        assert all(type(t) is int and 0 <= t < spec["vocab_size"] for t in generated)
        assert isinstance(row["prediction"], str)
        prefill = row["prefill_cache_tokens"]
        assert len(prefill) == len(row["final_cache_tokens"]) == len(row["cache_heads"]) == NUM_LAYERS
        assert all(0 < length <= prompt_len for length in prefill)
        assert row["final_cache_tokens"] == [length + len(generated) - 1 for length in prefill]
        assert all(heads in (NUM_Q_HEADS, NUM_KV_HEADS) for heads in row["cache_heads"])
        if mode == "fullkv":
            assert prefill == [prompt_len] * NUM_LAYERS
            assert row["cache_heads"] == [NUM_KV_HEADS] * NUM_LAYERS
        elif prompt_len > spec["budget"]:
            assert min(prefill) < prompt_len


def score_function():
    # Extract the unchanged, dependency-free TREC metric from the authors' pinned metrics.py.
    # See the pinned upstream repository and its MIT license.
    source = download_upstream("metrics.py", METRICS_SHA256)
    tree = ast.parse(source, filename="upstream/metrics.py")
    function = next(node for node in tree.body if isinstance(node, ast.FunctionDef)
                    and node.name == "classification_score")
    namespace = {}
    module = ast.Module(body=[function], type_ignores=[])
    exec(compile(module, "upstream/metrics.py", "exec"), namespace)
    return namespace["classification_score"]


def score_prediction(classifier, prediction, case):
    # The official eval.py scores the first line for TREC and takes the best reference score.
    first_line = prediction.lstrip("\n").split("\n", 1)[0]
    return max(classifier(first_line, answer, all_classes=case["all_classes"])
               for answer in case["answers"])


def make_zip(output, names):
    hashes = {name: digest((output / name).read_bytes()) for name in names}
    write_json(output / "integrity.json", {"file_sha256": hashes})
    path = output.with_suffix(".zip")
    temporary = path.with_suffix(".zip.tmp")
    with ZipFile(temporary, "w", compression=ZIP_DEFLATED) as archive:
        for name in [*names, "integrity.json"]:
            archive.write(output / name, name)
    temporary.replace(path)
    with ZipFile(path) as archive:
        assert archive.testzip() is None and set(archive.namelist()) == set(names) | {"integrity.json"}
    return path


def read_verified_zip(path, names):
    with ZipFile(path) as archive:
        expected = set(names) | {"integrity.json"}
        assert set(archive.namelist()) == expected and len(archive.namelist()) == len(expected)
        assert sum(member.file_size for member in archive.infolist()) < 100_000_000
        integrity = json.loads(archive.read("integrity.json"))["file_sha256"]
        assert set(integrity) == set(names)
        result = {name: archive.read(name) for name in names}
    assert all(digest(data) == integrity[name] for name, data in result.items())
    return result


OFFICIAL_FILES = ("workload.json", "cases.jsonl", "run_manifest.json", "fullkv.jsonl", "official_pyramidkv.jsonl")

In [4]:
GPU = observed_gpu()
assert transformers.__version__ == "5.12.1"
official_files = read_verified_zip(REFERENCE_ZIP, OFFICIAL_FILES)
SPEC = json.loads(official_files["workload.json"])
CASES = read_jsonl(official_files["cases.jsonl"])
validate_workload(SPEC, CASES)
WORKLOAD_SHA = object_digest({"spec": SPEC, "cases": CASES})
A_MANIFEST = json.loads(official_files["run_manifest.json"])
assert A_MANIFEST["schema"] == SCHEMA and A_MANIFEST["arm"] == "official"
assert A_MANIFEST["workload_sha256"] == WORKLOAD_SHA
assert A_MANIFEST["case_count"] == CASE_COUNT and A_MANIFEST["row_count"] == 2 * CASE_COUNT
assert A_MANIFEST["upstream_commit"] == UPSTREAM_COMMIT
assert A_MANIFEST["versions"]["transformers"] == "4.44.2"
assert A_MANIFEST["pilot"]["passed"] and "T4" in A_MANIFEST["gpu"]["name"]
assert A_MANIFEST["upstream_eviction"] == {
    "window_size": 8, "max_capacity_prompt": SPEC["budget"], "kernel_size": 7,
    "pooling": "maxpool", "merge": None, "kv_cache_granularity": "query_head",
}
assert SPEC["precision"] == PRECISION
REF_ROWS = {
    mode: read_jsonl(official_files[f"{mode}.jsonl"])
    for mode in ("fullkv", "official_pyramidkv")
}
for mode, rows in REF_ROWS.items():
    validate_rows(rows, CASES, mode, SPEC)
assert all(r["cache_heads"] == [NUM_Q_HEADS] * NUM_LAYERS for r in REF_ROWS["official_pyramidkv"])
TOKENIZER, b_stop_ids = tokenizer_and_stop_ids(SPEC)
assert b_stop_ids == SPEC["stop_token_ids"] and len(TOKENIZER) == SPEC["vocab_size"]
b_pad_id = TOKENIZER.pad_token_id if TOKENIZER.pad_token_id is not None else TOKENIZER.eos_token_id
assert b_pad_id == SPEC["pad_token_id"]
assert all(TOKENIZER.decode(row["generated_ids"], skip_special_tokens=True, clean_up_tokenization_spaces=False) == row["prediction"]
           for rows in REF_ROWS.values() for row in rows), "Token decoding differs between the runtimes."

OUTPUT, RUN_ID = start_output_directory("comparison")
a_file_names = {
    "workload.json": "workload.json",
    "cases.jsonl": "cases.jsonl",
    "run_manifest.json": "official_run_manifest.json",
    "fullkv.jsonl": "official_fullkv.jsonl",
    "official_pyramidkv.jsonl": "official_pyramidkv.jsonl",
}
for name, output_name in a_file_names.items():
    (OUTPUT / output_name).write_bytes(official_files[name])
del official_files
print(f"Verified all {len(CASES)} matching TREC inputs and both official runs.")
print(f"Workload SHA-256 {WORKLOAD_SHA}")
if package_versions()["torch"] != A_MANIFEST["versions"]["torch"]:
    print("PyTorch versions differ. The comparison report will record both.")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Verified all 200 matching TREC inputs and both official runs.
Workload SHA-256 94bd110fc970e318bb781284528fbfca667754f7fd1acf53a96b194a60de9ee1


## 3. Load one 3B model and run the pilot

The wrapper uses the paper’s 32-token observation window, average pooling, and native KV-head layout. The pilot checks actual cache lengths and verifies that turning the wrapper on with eviction disabled leaves generated tokens unchanged on the shortest and longest pilot prompts.

In [5]:
from contextlib import contextmanager

import torch.nn.functional as F
from transformers.modeling_utils import ALL_ATTENTION_FUNCTIONS
from transformers.models.llama.modeling_llama import (
    LlamaAttention, apply_rotary_pos_emb, eager_attention_forward, repeat_kv,
)

# These settings and the functions below come from the paper's Notebook 12.
WINDOW = 32
POOL_KERNEL = 5
PYRAMID_BETA = 20
DEFAULT_AVERAGE_BUDGET = SPEC["budget"]


def build_pyramid_budgets(
    layers=NUM_LAYERS,
    average_budget=DEFAULT_AVERAGE_BUDGET,
    window=WINDOW,
    beta=PYRAMID_BETA,
):
    average_budget = int(average_budget)
    assert layers > 1 and average_budget > window and beta > 0
    old_average = average_budget - window
    last = round(old_average / beta)
    first = 2 * old_average - last
    budgets = (np.linspace(first, last, layers).round().astype(int) + window).tolist()
    assert len(budgets) == layers and int(np.mean(budgets)) == average_budget
    assert min(budgets) >= window and all(a >= b for a, b in zip(budgets, budgets[1:]))
    return budgets


PYRAMID_BUDGETS = build_pyramid_budgets()
PKV_LOG = []
CACHE_UPDATE_LOG = []
ACTIVE_EVICTION = False
ORIGINAL_LLAMA_FORWARD = LlamaAttention.forward


def select_pyramid_kv(query_states, key_states, value_states, layer_idx, groups, scaling):
    cache_len = key_states.shape[-2]
    budget = min(PYRAMID_BUDGETS[layer_idx], cache_len)
    if cache_len <= budget:
        return key_states, value_states
    obs = min(WINDOW, cache_len)
    old_len = cache_len - obs
    old_keep = max(budget - obs, 0)
    recent_query = query_states[:, :, -obs:, :]
    repeated_key = repeat_kv(key_states, groups) if groups > 1 else key_states
    scores = torch.matmul(recent_query, repeated_key.transpose(2, 3)) * scaling
    causal = torch.triu(
        torch.full((obs, obs), float("-inf"), device=scores.device, dtype=scores.dtype),
        diagonal=1,
    )
    scores[:, :, :, -obs:] += causal[None, None, :, :]
    probabilities = F.softmax(scores, dim=-1, dtype=torch.float32)
    prefix_scores = probabilities[:, :, :, :old_len].sum(dim=-2)
    prefix_scores = prefix_scores.reshape(
        prefix_scores.shape[0], key_states.shape[1], groups, old_len,
    ).mean(dim=2)
    prefix_scores = F.avg_pool1d(
        prefix_scores, kernel_size=POOL_KERNEL, stride=1, padding=POOL_KERNEL // 2,
    )
    topk = prefix_scores.topk(old_keep, dim=-1).indices.sort(dim=-1).values
    gather_index = topk.unsqueeze(-1).expand(-1, -1, -1, key_states.shape[-1])
    selected_key = torch.cat(
        [key_states[:, :, :old_len, :].gather(2, gather_index), key_states[:, :, -obs:, :]],
        dim=2,
    ).contiguous()
    selected_value = torch.cat(
        [value_states[:, :, :old_len, :].gather(2, gather_index), value_states[:, :, -obs:, :]],
        dim=2,
    ).contiguous()
    PKV_LOG.append({
        "layer": int(layer_idx),
        "before": int(cache_len),
        "after": int(selected_key.shape[-2]),
        "budget": int(budget),
        "head_selection_differs": bool((topk[:, 1:] != topk[:, :-1]).any().item()),
    })
    return selected_key, selected_value


def cache_wrapper_forward(
    self,
    hidden_states,
    position_embeddings=None,
    attention_mask=None,
    past_key_values=None,
    **kwargs,
):
    input_shape = hidden_states.shape[:-1]
    hidden_shape = (*input_shape, -1, self.head_dim)
    query_states = self.q_proj(hidden_states).view(hidden_shape).transpose(1, 2)
    key_states = self.k_proj(hidden_states).view(hidden_shape).transpose(1, 2)
    value_states = self.v_proj(hidden_states).view(hidden_shape).transpose(1, 2)
    cos, sin = position_embeddings
    query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)
    is_prefill = (
        past_key_values is not None
        and past_key_values.get_seq_length(self.layer_idx) == 0
    )
    if past_key_values is not None and not is_prefill:
        input_tokens = int(key_states.shape[-2])
        key_states, value_states = past_key_values.update(
            key_states, value_states, self.layer_idx,
        )
        CACHE_UPDATE_LOG.append({
            "phase": "decode",
            "layer": int(self.layer_idx),
            "input_tokens": input_tokens,
            "returned_tokens": int(key_states.shape[-2]),
            "cache_type": type(past_key_values).__name__,
        })
    attention_tokens = int(key_states.shape[-2])
    attention_interface = ALL_ATTENTION_FUNCTIONS.get_interface(
        self.config._attn_implementation, eager_attention_forward,
    )
    attention_output, attention_weights = attention_interface(
        self,
        query_states,
        key_states,
        value_states,
        attention_mask,
        dropout=0.0,
        scaling=self.scaling,
        **kwargs,
    )
    if is_prefill:
        stored_key, stored_value = key_states, value_states
        if ACTIVE_EVICTION:
            stored_key, stored_value = select_pyramid_kv(
                query_states,
                key_states,
                value_states,
                self.layer_idx,
                self.num_key_value_groups,
                self.scaling,
            )
        update_input_tokens = int(stored_key.shape[-2])
        returned_key, returned_value = past_key_values.update(
            stored_key, stored_value, self.layer_idx,
        )
        CACHE_UPDATE_LOG.append({
            "phase": "prefill",
            "layer": int(self.layer_idx),
            "attention_tokens": attention_tokens,
            "update_input_tokens": update_input_tokens,
            "returned_tokens": int(returned_key.shape[-2]),
            "cache_type": type(past_key_values).__name__,
        })
        del returned_key, returned_value
    attention_output = attention_output.reshape(*input_shape, -1).contiguous()
    return self.o_proj(attention_output), attention_weights


@contextmanager
def attention_mode(mode):
    global ACTIVE_EVICTION
    assert mode in ("fullkv", "wrapped_fullkv", "custom_pyramidkv")
    assert LlamaAttention.forward is ORIGINAL_LLAMA_FORWARD
    ACTIVE_EVICTION = mode == "custom_pyramidkv"
    if mode != "fullkv":
        LlamaAttention.forward = cache_wrapper_forward
    try:
        yield
    finally:
        LlamaAttention.forward = ORIGINAL_LLAMA_FORWARD
        ACTIVE_EVICTION = False
        CACHE_UPDATE_LOG.clear()
        PKV_LOG.clear()


@torch.inference_mode()
def generate_case(model, case, mode, tokenizer, spec):
    PKV_LOG.clear()
    CACHE_UPDATE_LOG.clear()
    input_ids = torch.tensor([case["input_ids"]], dtype=torch.long, device=model.device)
    prompt_len = input_ids.shape[-1]
    cache = DynamicCache()
    output = model(input_ids=input_ids, past_key_values=cache, use_cache=True, logits_to_keep=1)
    kept = cache_shapes(cache)
    next_token = output.logits[:, -1, :].argmax(-1, keepdim=True)
    generated = []
    for step in range(spec["max_new_tokens"]):
        token = int(next_token.item())
        generated.append(token)
        if token in spec["stop_token_ids"] or step == spec["max_new_tokens"] - 1:
            break
        absolute_position = prompt_len + step
        position = torch.tensor([[absolute_position]], device=model.device)
        output = model(
            input_ids=next_token, position_ids=position, cache_position=position[0],
            past_key_values=cache, use_cache=True, logits_to_keep=1,
        )
        next_token = output.logits[:, -1, :].argmax(-1, keepdim=True)
    if mode == "custom_pyramidkv":
        expected = [min(prompt_len, budget) for budget in PYRAMID_BUDGETS]
        assert kept == [(NUM_KV_HEADS, length) for length in expected]
        prefill = [entry for entry in CACHE_UPDATE_LOG if entry["phase"] == "prefill"]
        assert [entry["layer"] for entry in prefill] == list(range(NUM_LAYERS))
        assert all(entry["attention_tokens"] == prompt_len for entry in prefill)
        assert [entry["returned_tokens"] for entry in prefill] == expected
        decode = [entry for entry in CACHE_UPDATE_LOG if entry["phase"] == "decode"]
        assert len(decode) == NUM_LAYERS * (len(generated) - 1)
        assert all(entry["input_tokens"] == 1 for entry in decode)
        assert [entry["layer"] for entry in PKV_LOG] == [
            layer for layer, budget in enumerate(PYRAMID_BUDGETS) if prompt_len > budget
        ]
    return build_row(case, mode, generated, kept, cache_shapes(cache), tokenizer, spec)


MODEL = load_model(SPEC, "5.12.1")
assert all(isinstance(layer.self_attn, LlamaAttention) for layer in MODEL.model.layers)
print(f"Loaded {PRECISION} Llama 3.2-3B on {GPU['name']} with the custom SDPA wrapper available.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.25GB            

model.safetensors: downloading bytes:           |  0.00B            


INFO  LogBar: headless/CI mode; progress animations disabled.                   
WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.
INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.
INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.           
INFO    ┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐ │ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │ └─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘ GPT-QModel   : 7.1.0 Transformers : 5.12.1 Torch        : 2.11.0+cu128 Triton       : 3.6.0
INFO  ExLlamaV2 AWQ: compiling torch.ops JIT extension in `/root/.cache/gptqmodel/torch_

Loading weights:   0%|          | 0/646 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

INFO  gc.collect() reclaimed 5658 objects in 0.370s                             
Loaded awq Llama 3.2-3B on Tesla T4 with the custom SDPA wrapper available.


In [6]:
def run_cases(cases, mode):
    results = []
    phase_dir = OUTPUT / "rows" / mode
    phase_dir.mkdir(parents=True, exist_ok=True)
    with attention_mode(mode):
        for offset, case in enumerate(cases, 1):
            row_path = phase_dir / f'{case["index"]:03d}.json'
            if row_path.is_file():
                row = read_json(row_path)
                validate_rows([row], [case], mode, SPEC)
            else:
                row = generate_case(MODEL, case, mode, TOKENIZER, SPEC)
                write_json(row_path, row)
            results.append(row)
            if offset == 1 or offset % 20 == 0 or offset == len(cases):
                print(f"{mode}: {offset}/{len(cases)}", flush=True)
    validate_rows(results, cases, mode, SPEC)
    return results


def run_pilot(variants):
    samples = pilot_cases(CASES)
    durations = {}
    for mode in variants:
        torch.cuda.reset_peak_memory_stats()
        start = time.perf_counter()
        with attention_mode(mode):
            results = [generate_case(MODEL, case, mode, TOKENIZER, SPEC) for case in samples]
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        validate_rows(results, samples, mode, SPEC)
        if mode != "fullkv":
            assert any(min(row["prefill_cache_tokens"]) < len(case["input_ids"])
                       for row, case in zip(results, samples)), "Pilot observed no eviction"
        durations[mode] = {
            "wall_seconds": round(elapsed, 3),
            "peak_allocated_mib": round(torch.cuda.max_memory_allocated() / 2**20, 1),
        }
    info = {"case_indices": [case["index"] for case in samples],
            "prompt_lengths": [len(case["input_ids"]) for case in samples],
            "by_mode": durations, "passed": True}
    write_json(OUTPUT / "pilot.json", info)
    print(json.dumps(info, indent=2))
    return info

MODES = ("fullkv", "custom_pyramidkv")
PILOT = run_pilot(MODES)
# Compare stock and wrapped FullKV within the same runtime with eviction disabled.
parity_cases = pilot_cases(CASES)[::4]
parity = []
for case in parity_cases:
    with attention_mode("fullkv"):
        stock = generate_case(MODEL, case, "fullkv", TOKENIZER, SPEC)
    with attention_mode("wrapped_fullkv"):
        wrapped = generate_case(MODEL, case, "fullkv", TOKENIZER, SPEC)
    parity.append({
        "case_index": case["index"],
        "same_generated_tokens": stock["generated_ids"] == wrapped["generated_ids"],
    })
PILOT["wrapped_fullkv_parity"] = parity
PILOT["passed"] = all(row["same_generated_tokens"] for row in parity)
write_json(OUTPUT / "pilot.json", PILOT)
assert PILOT["passed"], "Wrapper-off parity check failed. Share this cell output before running the full benchmark."
print("Stock and non-evicting wrapped FullKV generated identical tokens on both parity prompts.")

{
  "case_indices": [
    58,
    38,
    184,
    96,
    116
  ],
  "prompt_lengths": [
    1884,
    4096,
    4096,
    4096,
    4096
  ],
  "by_mode": {
    "fullkv": {
      "wall_seconds": 45.749,
      "peak_allocated_mib": 6558.6
    },
    "custom_pyramidkv": {
      "wall_seconds": 36.53,
      "peak_allocated_mib": 6243.0
    }
  },
  "passed": true
}
Stock and non-evicting wrapped FullKV generated identical tokens on both parity prompts.


## 4. Run the benchmark and compare the results

Run both B paths on all 200 prompts. Compute the authors’ TREC score and paired percentage-point differences. Compare the two FullKV controls to assess baseline agreement across software environments. Use the reported cache retention to interpret differences at the same nominal budget.

If interrupted while the runtime remains connected, rerun this cell to resume. It also writes a Markdown summary, per-case scores, and raw outputs for all four configurations.

In [7]:
import csv

assert PILOT["passed"]
B_ROWS = {mode: run_cases(CASES, mode) for mode in MODES}
write_jsonl(OUTPUT / "custom_fullkv.jsonl", B_ROWS["fullkv"])
write_jsonl(OUTPUT / "custom_pyramidkv.jsonl", B_ROWS["custom_pyramidkv"])

manifest = {
    "schema": SCHEMA, "arm": "custom", "run_id": RUN_ID, "workload_sha256": WORKLOAD_SHA,
    "official_archive_sha256": digest(REFERENCE_ZIP.read_bytes()),
    "gpu": GPU, "versions": package_versions(), "model_runtime": model_runtime(MODEL),
    "case_count": CASE_COUNT, "row_count": len(MODES) * CASE_COUNT, "pilot": PILOT,
    "custom_eviction": {
        "implementation": "12_llama_matched_wrapper_interaction_v1.ipynb",
        "window": WINDOW, "pool_kernel": POOL_KERNEL, "pooling": "avgpool",
        "beta": PYRAMID_BETA, "layer_budgets": PYRAMID_BUDGETS, "kv_cache_granularity": "kv_head",
    },
}
write_json(OUTPUT / "custom_run_manifest.json", manifest)

all_rows = {
    "official_fullkv": REF_ROWS["fullkv"],
    "official_pyramidkv": REF_ROWS["official_pyramidkv"],
    "custom_fullkv": B_ROWS["fullkv"],
    "custom_pyramidkv": B_ROWS["custom_pyramidkv"],
}
classifier = score_function()
scores = {
    mode: np.array([score_prediction(classifier, row["prediction"], case)
                    for row, case in zip(rows, CASES)], dtype=float)
    for mode, rows in all_rows.items()
}
contrasts = {
    "official_eviction_minus_its_fullkv": scores["official_pyramidkv"] - scores["official_fullkv"],
    "custom_eviction_minus_its_fullkv": scores["custom_pyramidkv"] - scores["custom_fullkv"],
    "custom_minus_official_eviction": scores["custom_pyramidkv"] - scores["official_pyramidkv"],
    "custom_minus_official_fullkv": scores["custom_fullkv"] - scores["official_fullkv"],
}
contrasts["custom_minus_official_change_from_fullkv"] = (
    contrasts["custom_eviction_minus_its_fullkv"] - contrasts["official_eviction_minus_its_fullkv"]
)
bootstrap_draws = 20_000
rng = np.random.default_rng(SEED)
paired_resamples = rng.integers(0, CASE_COUNT, size=(bootstrap_draws, CASE_COUNT))


def summarize_contrast(deltas):
    low, high = np.quantile(deltas[paired_resamples].mean(axis=1), [0.025, 0.975]) * 100
    return {"difference_pp": float(deltas.mean() * 100), "ci95_pp": [float(low), float(high)]}


def cache_summary(rows):
    return {
        "mean_prefill_tokens_per_layer": float(np.mean([r["prefill_cache_tokens"] for r in rows])),
        "mean_prefill_head_token_slots": float(np.mean([
            sum(heads * tokens for heads, tokens in zip(r["cache_heads"], r["prefill_cache_tokens"]))
            for r in rows
        ])),
        "cache_heads_per_layer": sorted({heads for r in rows for heads in r["cache_heads"]}),
    }


results = {
    "schema": SCHEMA, "workload_sha256": WORKLOAD_SHA, "workload": SPEC,
    "official_versions": A_MANIFEST["versions"], "custom_versions": manifest["versions"],
    "case_count": CASE_COUNT, "bootstrap_draws": bootstrap_draws, "bootstrap_seed": SEED,
    "truncated_prompt_count": sum(case["original_token_count"] > SPEC["prompt_token_cap"] for case in CASES),
    "score_pct": {mode: float(values.mean() * 100) for mode, values in scores.items()},
    "paired_contrasts": {name: summarize_contrast(delta) for name, delta in contrasts.items()},
    "fullkv_same_generated_ids_count": sum(
        a["generated_ids"] == b["generated_ids"]
        for a, b in zip(all_rows["official_fullkv"], all_rows["custom_fullkv"])
    ),
    "fullkv_same_score_count": int(np.count_nonzero(scores["official_fullkv"] == scores["custom_fullkv"])),
    "cache": {mode: cache_summary(rows) for mode, rows in all_rows.items()},
}
write_json(OUTPUT / "comparison.json", results)
with (OUTPUT / "paired_scores.csv").open("w", encoding="utf-8", newline="") as handle:
    writer = csv.writer(handle)
    writer.writerow(["case_id", "prompt_tokens", "original_prompt_tokens", *scores])
    for index, case in enumerate(CASES):
        writer.writerow([case["case_id"], len(case["input_ids"]), case["original_token_count"],
                         *(float(values[index]) for values in scores.values())])

method_names = {
    "official_fullkv": "FullKV (upstream environment)",
    "official_pyramidkv": "Authors’ PyramidKV",
    "custom_fullkv": "FullKV (custom environment)",
    "custom_pyramidkv": "Paper’s custom PyramidKV-style eviction",
}
contrast_names = {
    "official_eviction_minus_its_fullkv": "Authors’ PyramidKV minus its FullKV",
    "custom_eviction_minus_its_fullkv": "Custom eviction minus its FullKV",
    "custom_minus_official_eviction": "Custom eviction minus authors’ PyramidKV",
    "custom_minus_official_fullkv": "Custom FullKV minus upstream FullKV",
    "custom_minus_official_change_from_fullkv": "Difference in changes from each FullKV (custom minus upstream)",
}
lines = [
    "# PyramidKV validation on LongBench v1 TREC", "",
    f"All {CASE_COUNT} TREC examples used identical token IDs, a single pinned Llama 3.2-3B-Instruct",
    f"{SPEC['precision'].upper()} checkpoint, greedy decoding, and a {SPEC['prompt_token_cap']:,}-token prompt cap.",
    f"Each eviction implementation used nominal K={SPEC['budget']:,}. Cache settings and head layouts",
    "follow each implementation. Counts below refer to the prefill cache and exclude decode tokens.", "",
    "| Configuration | TREC score (%) | Mean tokens per layer | Mean head-token slots across layers |",
    "|---|---:|---:|---:|",
]
for mode, score in results["score_pct"].items():
    cache = results["cache"][mode]
    lines.append(f"| {method_names[mode]} | {score:.2f} | {cache['mean_prefill_tokens_per_layer']:.1f} | "
                 f"{cache['mean_prefill_head_token_slots']:.0f} |")
lines += [
    "", "| Paired contrast | Difference (pp) | 95% paired bootstrap CI (pp) |",
    "|---|---:|---:|",
]
for name, values in results["paired_contrasts"].items():
    low, high = values["ci95_pp"]
    lines.append(f"| {contrast_names[name]} | {values['difference_pp']:+.2f} | [{low:+.2f}, {high:+.2f}] |")
lines += [
    "", f"Stock FullKV generated identical token sequences on "
    f"{results['fullkv_same_generated_ids_count']} of {CASE_COUNT} cases and had identical per-case "
    f"scores on {results['fullkv_same_score_count']} cases.",
    "", f"The upstream path used Transformers {A_MANIFEST['versions']['transformers']} and PyTorch "
    f"{A_MANIFEST['versions']['torch']}. The custom path used Transformers "
    f"{manifest['versions']['transformers']} and PyTorch {manifest['versions']['torch']}. The two FullKV "
    "controls measure baseline agreement across software environments. Differences can still reflect how "
    "eviction interacts with the software stack. This comparison establishes the observed score gap "
    "on the 3B TREC setup. Reproducing the published Llama-3-8B scores requires the published setup.", "",
]
(OUTPUT / "reviewer_summary.md").write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines))
FINAL_FILES = (
    *a_file_names.values(), "custom_run_manifest.json", "custom_fullkv.jsonl", "custom_pyramidkv.jsonl",
    "comparison.json", "paired_scores.csv", "reviewer_summary.md",
)
COMPARISON_ZIP = make_zip(OUTPUT, FINAL_FILES)
print(f"Final comparison ZIP: {COMPARISON_ZIP}")

fullkv: 1/200
fullkv: 20/200
fullkv: 40/200
fullkv: 60/200
fullkv: 80/200
fullkv: 100/200
fullkv: 120/200
fullkv: 140/200
fullkv: 160/200
fullkv: 180/200
fullkv: 200/200
custom_pyramidkv: 1/200
custom_pyramidkv: 20/200
custom_pyramidkv: 40/200
custom_pyramidkv: 60/200
custom_pyramidkv: 80/200
custom_pyramidkv: 100/200
custom_pyramidkv: 120/200
custom_pyramidkv: 140/200
custom_pyramidkv: 160/200
custom_pyramidkv: 180/200
custom_pyramidkv: 200/200
# PyramidKV validation on LongBench v1 TREC

All 200 TREC examples used identical token IDs, a single pinned Llama 3.2-3B-Instruct
AWQ checkpoint, greedy decoding, and a 4,096-token prompt cap.
Each eviction implementation used nominal K=1,024. Cache settings and head layouts
follow each implementation. Counts below refer to the prefill cache and exclude decode tokens.

| Configuration | TREC score (%) | Mean tokens per layer | Mean head-token slots across layers |
|---|---:|---:|---:|
| FullKV (upstream environment) | 67.50 | 3848.5 | 862057 |

## 5. Download the final comparison ZIP

Share the `paper744_trec_comparison_*.zip` file. If the browser blocks the download, retrieve it from the Colab Files sidebar.

In [8]:
from google.colab import files

files.download(str(COMPARISON_ZIP))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>